# Hierarchical Hybrid Mamba: Dual-Lookback Continuous Intraday Optimization

This notebook trains **DualLookbackHierarchicalMamba** on continuous 5-minute bar data using:
- **ContinuousIntradayPrep** for feature engineering (deseasonalized vol/volume, overnight returns)
- **Dual lookback architecture**: Long branch (256 bars) + Short branch (32-128 bars)
- **3-way classification** (terciles of 60-minute forward returns)
- **ProfitWeightedCE** loss for trading-aware optimization
- **Accuracy-weighted PnL** optimization objective

## Architecture Overview

**DualLookbackHierarchicalMamba** uses dual-scale processing with separate lookbacks:
- **Long Branch** (256 bars → 21 tokens): StandardMambaBlock with d_state=64 for structural trends
  - Patch size: 12 bars (60min), stride: 12 (non-overlapping)
  - Token count: 256 / 12 = 21 tokens
- **Short Branch** (32-128 bars → 31-127 tokens): CMDMambaBlock for immediate volatility
  - Patch size: 2 bars (10min), stride: 1 (overlapping)
  - Uses only recent data for fine-grained signals
- **Fusion**: Concatenate last states from both branches

## Optimization Objective

```
score = pnl * (accuracy / 100) * max(0, accuracy - baseline)
```

Rewards models that are profitable, accurate, and beat random guessing.

## 1. Environment Setup

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:

    IN_COLAB = False
    print("Not running in Colab - skipping drive mount")

In [ ]:
# Add CTAFlow to path (Colab only)
if IN_COLAB:
    import sys
    %cd /content/drive/MyDrive/CTAEnv/
    %cd CTAFlow
    !git pull
    %cd ..
    sys.path.insert(0, '/content/drive/MyDrive/CTAEnv/CTAFlow/')
    sys.path.insert(1, '/content/drive/MyDrive/CTAEnv/SierraPy')
    !pip install -e SierraPy -q
    !pip install -e CTAFlow -q
    !pip install optuna mamba_ssm causal-conv1d -q
else:
    !cd /workspace/CTAFlow && git pull
    %cd /workspace/
    !pip install -e CTAFlow
    !pip install -e SierraPy -q
    sys.path.insert(0, '/workspace/CTAFlow')
    sys.path.insert(1, '/workspace/SierraPy')
    !pip install optuna
    !pip install mamba_ssm --no-build-isolation
    print("Running locally - ensure CTAFlow, optuna, and mamba_ssm are installed")

In [ ]:
import json
import warnings
from datetime import time, timedelta
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Optuna version: {optuna.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# Check mamba availability
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print("mamba_ssm: Available")
except ImportError:
    MAMBA_AVAILABLE = False
    print("mamba_ssm: Not installed - this notebook requires mamba_ssm!")

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set seeds for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 2. Configuration

In [ ]:
# --- Ticker & Data Configuration ---
TICKER = 'ES'  # E-mini S&P 500 (or your ticker)
BAR_MINUTES = 5  # 5-minute bars

# --- Session Configuration (US Equity) ---
# Adjust for your market
SESSION_OPEN = "08:30"  # 8:30 AM CT
SESSION_END = "15:00"   # 3:00 PM CT (before close)
TIMEZONE = 'America/Chicago'

# --- Target Configuration ---
TASK = 'classification'
NUM_CLASSES = 3
TARGET_STEPS = 12  # 12 * 5min = 60min forward
CLF_PERCENTILES = (33, 67)  # Terciles

# --- Dual Lookback Configuration ---
# Long branch: full structural context
LONG_LOOKBACK = 256  # 256 * 5min = ~21 hours
LONG_PATCH = 12      # 12 bars = 60min patches -> 256/12 = 21 tokens

# Short branch: recent fine-grained signals (tunable 32-128)
SHORT_LOOKBACK_MIN = 32   # Minimum short lookback
SHORT_LOOKBACK_MAX = 128  # Maximum short lookback
SHORT_PATCH = 2           # 2 bars = 10min patches

print(f"Ticker: {TICKER}")
print(f"Bar frequency: {BAR_MINUTES} minutes")
print(f"Session: {SESSION_OPEN} - {SESSION_END} ({TIMEZONE})")
print(f"Target: {TARGET_STEPS * BAR_MINUTES}min forward returns, {NUM_CLASSES} classes")
print(f"\nDual Lookback Architecture:")
print(f"  Long branch:  {LONG_LOOKBACK} bars ({LONG_LOOKBACK * BAR_MINUTES / 60:.1f}h) -> {LONG_LOOKBACK // LONG_PATCH} tokens")
print(f"  Short branch: {SHORT_LOOKBACK_MIN}-{SHORT_LOOKBACK_MAX} bars (tunable)")

In [ ]:
# Path configuration
if IN_COLAB:
    DRIVE_PATH = Path('/content/drive/MyDrive')
else:
    DRIVE_PATH = Path.cwd().parent

# Path to your 5-minute OHLCV data (CSV with DatetimeIndex)
DATA_PATH = DRIVE_PATH / 'data' / f'{TICKER}_5min.csv'
RESULTS_PATH = DRIVE_PATH / 'results' / f'hierarchical_mamba_{TICKER}_optuna'
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Results path: {RESULTS_PATH}")

## 3. Load & Prepare Data

In [ ]:
from CTAFlow.models.prep.intraday_continuous import (
    ContinuousIntradayPrep,
    SessionSpec,
)

# Load raw data
print(f"Loading data from {DATA_PATH}...")
raw_df = pd.read_csv(DATA_PATH, parse_dates=True, index_col=0)

# Standardize column names
col_map = {}
for c in raw_df.columns:
    cl = c.lower()
    if cl == 'open': col_map[c] = 'Open'
    elif cl == 'high': col_map[c] = 'High'
    elif cl == 'low': col_map[c] = 'Low'
    elif cl in ('close', 'last'): col_map[c] = 'Close'
    elif cl in ('volume', 'vol'): col_map[c] = 'Volume'
raw_df = raw_df.rename(columns=col_map)

print(f"Loaded {len(raw_df):,} bars")
print(f"Date range: {raw_df.index.min()} to {raw_df.index.max()}")
print(f"Columns: {list(raw_df.columns)}")

In [ ]:
# Configure feature preparation
prep = ContinuousIntradayPrep(
    sessions=[
        SessionSpec("USA", SESSION_OPEN, SESSION_END),
    ],
    bar_minutes=BAR_MINUTES,
)

print("Preparing features...")
print("This may take a few minutes for deseasonalization.\n")

df_prepared, train_mask, target_cols = prep.prepare(
    raw_df,
    steps_60m=TARGET_STEPS,
    keep_only_active=False,  # Keep full timeline for proper windowing
    add_daily=True,
    add_overnight=True,
    add_deseas=True,
    rolling_days_deseas=252,
    refit_interval=10,
    use_legacy_deseas=False,
)

print(f"Prepared {len(df_prepared):,} bars")
print(f"Valid training samples: {train_mask.sum():,}")
print(f"Target columns: {target_cols}")

In [ ]:
# Define feature columns (exclude targets, metadata)
EXCLUDE_COLS = set(target_cols + [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'session_code', 'is_active', 'is_usa', 'is_london',
    'tod_slot', 'log_ret',
])

FEATURE_COLS = [
    c for c in df_prepared.columns 
    if c not in EXCLUDE_COLS and df_prepared[c].dtype in [np.float32, np.float64, np.int32, np.int64]
]

print(f"Feature columns ({len(FEATURE_COLS)}):")
for i, c in enumerate(FEATURE_COLS):
    print(f"  {i+1:2}. {c}")

In [ ]:
# Create classification targets (terciles of 60min forward returns)
# Use the final target column (y_fwd_12 = 60min return for 5min bars)
TARGET_COL = target_cols[-1]  # y_fwd_12
print(f"Using target: {TARGET_COL}")

# Compute tercile thresholds from training data only
train_returns = df_prepared.loc[train_mask, TARGET_COL].dropna()
q_low = np.percentile(train_returns, CLF_PERCENTILES[0])
q_high = np.percentile(train_returns, CLF_PERCENTILES[1])

print(f"\nTercile thresholds:")
print(f"  Down (0): return < {q_low:.6f}")
print(f"  Neutral (1): {q_low:.6f} <= return < {q_high:.6f}")
print(f"  Up (2): return >= {q_high:.6f}")

# Create classification target
def classify_return(r):
    if pd.isna(r):
        return np.nan
    if r < q_low:
        return 0  # Down
    elif r < q_high:
        return 1  # Neutral
    else:
        return 2  # Up

df_prepared['target_class'] = df_prepared[TARGET_COL].apply(classify_return)

# Verify distribution
class_counts = df_prepared.loc[train_mask, 'target_class'].value_counts().sort_index()
print(f"\nClass distribution (training):")
for cls, count in class_counts.items():
    pct = count / class_counts.sum() * 100
    print(f"  Class {int(cls)}: {count:,} ({pct:.1f}%)")

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Returns distribution
ax = axes[0]
train_returns.hist(bins=50, ax=ax, alpha=0.7, edgecolor='black')
ax.axvline(q_low, color='red', linestyle='--', label=f'33rd pct: {q_low:.4f}')
ax.axvline(q_high, color='green', linestyle='--', label=f'67th pct: {q_high:.4f}')
ax.set_xlabel(f'{TARGET_COL} (log return)')
ax.set_ylabel('Count')
ax.set_title('60min Forward Return Distribution')
ax.legend()

# Class distribution
ax = axes[1]
bars = ax.bar(
    ['Down (0)', 'Neutral (1)', 'Up (2)'],
    class_counts.values,
    color=['#e74c3c', '#95a5a6', '#27ae60']
)
ax.set_ylabel('Count')
ax.set_title('Target Class Distribution')
for bar, count in zip(bars, class_counts.values):
    pct = count / class_counts.sum() * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 4. Dataset & DataLoader

In [ ]:
class ContinuousClassificationDataset(Dataset):
    """
    Dataset for continuous intraday classification.
    
    Returns:
        x: [lookback, n_features] - feature sequence
        y: int - class label (0, 1, 2)
        raw_return: float - raw return for PnL calculation
    """
    
    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols: List[str],
        target_col: str,
        class_col: str,
        lookback: int = 256,
        sample_mode: str = "active",
        enforce_contiguous: bool = True,
        bar_minutes: int = 5,
    ):
        self.df = df
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.class_col = class_col
        self.lookback = int(lookback)
        
        # Build sample mask
        n = len(df)
        ok = np.ones(n, dtype=bool)
        
        # Must have valid class and return
        ok &= df[class_col].notna().to_numpy()
        ok &= df[target_col].notna().to_numpy()
        
        # Must have enough history
        ok[:lookback - 1] = False
        
        # Session filtering
        if sample_mode == "active" and "is_active" in df.columns:
            ok &= df["is_active"].astype(bool).to_numpy()
        elif sample_mode == "active" and "session_code" in df.columns:
            ok &= (df["session_code"].to_numpy() > 0)
        
        # Contiguity check
        if enforce_contiguous:
            ok &= self._contiguous_check(df.index, lookback, bar_minutes)
        
        self.sample_pos = np.flatnonzero(ok)
        if len(self.sample_pos) == 0:
            raise ValueError("No eligible samples found!")
        
        # Cache arrays
        self.X = df[feature_cols].to_numpy(dtype=np.float32)
        self.Y_class = df[class_col].to_numpy(dtype=np.int64)
        self.Y_return = df[target_col].to_numpy(dtype=np.float32)
        
        # Handle NaN in features
        self.X = np.nan_to_num(self.X, nan=0.0, posinf=0.0, neginf=0.0)
    
    def _contiguous_check(self, idx: pd.DatetimeIndex, lookback: int, bar_minutes: int) -> np.ndarray:
        """Check for contiguous windows."""
        n = len(idx)
        ok = np.ones(n, dtype=bool)
        ok[:lookback - 1] = False
        
        diffs = idx.to_series().diff().dt.total_seconds().div(60.0).to_numpy()
        diffs[0] = bar_minutes
        
        good_step = (diffs == bar_minutes)
        bad = (~good_step).astype(np.int32)
        cbad = np.cumsum(bad)
        
        for i in range(lookback - 1, n):
            left = i - lookback + 1
            bad_in_window = cbad[i] - (cbad[left - 1] if left > 0 else 0)
            ok[i] = (bad_in_window == 0)
        
        return ok
    
    def __len__(self) -> int:
        return len(self.sample_pos)
    
    def __getitem__(self, idx: int):
        i = self.sample_pos[idx]
        j0 = i - self.lookback + 1
        
        x = torch.from_numpy(self.X[j0:i + 1].copy())  # [L, F]
        y_class = int(self.Y_class[i])
        y_return = float(self.Y_return[i])
        
        return x, y_class, y_return


def collate_fn(batch):
    """Custom collate to handle returns."""
    xs, ys, returns = zip(*batch)
    x = torch.stack(xs, dim=0)
    y = torch.tensor(ys, dtype=torch.long)
    r = torch.tensor(returns, dtype=torch.float32)
    return x, y, r

In [ ]:
def create_dataloaders(
    df: pd.DataFrame,
    feature_cols: List[str],
    long_lookback: int,
    batch_size: int,
    val_split: float = 0.2,
    enforce_contiguous: bool = True,
) -> Tuple[DataLoader, DataLoader]:
    """Create train/val dataloaders with time-based split.
    
    Uses long_lookback for data loading (the model handles short lookback internally).
    """
    
    # Time-based split to avoid lookahead
    dates = df.index.normalize().unique()
    split_idx = int(len(dates) * (1 - val_split))
    split_date = dates[split_idx]
    
    train_df = df[df.index < split_date].copy()
    val_df = df[df.index >= split_date].copy()
    
    print(f"Train: {len(train_df):,} bars, Val: {len(val_df):,} bars")
    print(f"Split date: {split_date}")
    
    train_ds = ContinuousClassificationDataset(
        df=train_df,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        class_col='target_class',
        lookback=long_lookback,  # Use long lookback for data loading
        sample_mode='active',
        enforce_contiguous=enforce_contiguous,
        bar_minutes=BAR_MINUTES,
    )
    
    val_ds = ContinuousClassificationDataset(
        df=val_df,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        class_col='target_class',
        lookback=long_lookback,
        sample_mode='active',
        enforce_contiguous=enforce_contiguous,
        bar_minutes=BAR_MINUTES,
    )
    
    print(f"Train samples: {len(train_ds):,}, Val samples: {len(val_ds):,}")
    
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True,
    )
    
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True,
    )
    
    return train_loader, val_loader

## 5. Model Definition

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.ms_mamba import (
    DualLookbackHierarchicalMamba,
    CMDMambaBlock,
    StandardMambaBlock,
)

# Print architecture info
print("DualLookbackHierarchicalMamba Architecture:")
print("=" * 50)
print(f"Long Branch:  {LONG_LOOKBACK} bars -> {LONG_LOOKBACK // LONG_PATCH} tokens")
print(f"  - Patch size: {LONG_PATCH} bars ({LONG_PATCH * BAR_MINUTES}min)")
print(f"  - StandardMambaBlock layers (d_state=64)")
print(f"\nShort Branch: {SHORT_LOOKBACK_MIN}-{SHORT_LOOKBACK_MAX} bars (tunable)")
print(f"  - Patch size: {SHORT_PATCH} bars ({SHORT_PATCH * BAR_MINUTES}min)")
print(f"  - CMDMambaBlock layers (d_state=16) with conv filtering")
print(f"\nFusion: Concatenate last states -> Classification head")

# Test model instantiation
test_model = DualLookbackHierarchicalMamba(
    input_dim=len(FEATURE_COLS),
    d_model=128,
    long_lookback=LONG_LOOKBACK,
    long_patch=LONG_PATCH,
    short_lookback=64,
    short_patch=SHORT_PATCH,
    num_classes=NUM_CLASSES,
)
print(f"\nTest model parameters: {sum(p.numel() for p in test_model.parameters()):,}")
print(f"Token counts: {test_model.get_token_counts()}")
del test_model

## 6. Training Functions

In [ ]:
from CTAFlow.models.deep_learning.training.loss import ProfitWeightedCE


def compute_accuracy_weighted_pnl(
    accuracy: float,
    pnl: float,
    max_class_pct: float,
    acc_weight: float = 1.0,
    baseline_margin_weight: float = 1.0,
) -> float:
    """
    Compute accuracy-weighted PnL score.
    
    score = pnl * (accuracy / 100)^w1 * (1 + margin/100)^w2
    
    With additional penalty if accuracy < baseline.
    """
    acc_term = (accuracy / 100.0) ** acc_weight
    margin = max(0.0, accuracy - max_class_pct)
    margin_term = (1.0 + margin / 100.0) ** baseline_margin_weight
    
    score = pnl * acc_term * margin_term
    
    if accuracy < max_class_pct:
        score = score * (accuracy / max_class_pct)
    
    return score


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    total_pnl = 0.0
    
    for x, y, returns in loader:
        x = x.to(device)
        y = y.to(device)
        returns = returns.to(device)
        
        optimizer.zero_grad()
        outputs = model(x)
        
        if isinstance(criterion, ProfitWeightedCE):
            loss = criterion(outputs, y, returns=returns)
        else:
            loss = criterion(outputs, y)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        total_samples += y.size(0)
        
        _, predicted = outputs.max(1)
        total_correct += predicted.eq(y).sum().item()
        
        # PnL: long if pred=2, short if pred=0, flat if pred=1
        probs = torch.softmax(outputs, dim=-1)
        position = probs[:, 2] - probs[:, 0]
        pnl = position * returns
        total_pnl += pnl.sum().item()
    
    avg_loss = total_loss / len(loader)
    acc = 100.0 * total_correct / total_samples
    avg_pnl = total_pnl / total_samples
    
    return avg_loss, acc, avg_pnl


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    total_pnl = 0.0
    
    all_preds = []
    all_targets = []
    all_returns = []
    
    with torch.no_grad():
        for x, y, returns in loader:
            x = x.to(device)
            y = y.to(device)
            returns = returns.to(device)
            
            outputs = model(x)
            
            if isinstance(criterion, ProfitWeightedCE):
                loss = criterion(outputs, y, returns=returns)
            else:
                loss = criterion(outputs, y)
            
            total_loss += loss.item()
            total_samples += y.size(0)
            
            _, predicted = outputs.max(1)
            total_correct += predicted.eq(y).sum().item()
            
            probs = torch.softmax(outputs, dim=-1)
            position = probs[:, 2] - probs[:, 0]
            pnl = position * returns
            total_pnl += pnl.sum().item()
            
            all_preds.append(outputs.cpu())
            all_targets.append(y.cpu())
            all_returns.append(returns.cpu())
    
    avg_loss = total_loss / len(loader)
    acc = 100.0 * total_correct / total_samples
    avg_pnl = total_pnl / total_samples
    
    all_targets = torch.cat(all_targets).numpy()
    unique, counts = np.unique(all_targets, return_counts=True)
    max_class_pct = 100.0 * counts.max() / len(all_targets)
    
    return avg_loss, acc, avg_pnl, max_class_pct

## 7. Optuna Optimization

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """
    Optuna objective using accuracy-weighted PnL with dual lookbacks.
    """
    # --- Architecture Hyperparameters ---
    # Long lookback is fixed at 256, short lookback is tunable
    short_lookback = trial.suggest_categorical('short_lookback', [32, 48, 64, 96, 128])
    
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    d_model = trial.suggest_categorical('d_model', [64, 128, 256])
    n_long_layers = trial.suggest_int('n_long_layers', 2, 4)
    n_short_layers = trial.suggest_int('n_short_layers', 1, 3)
    d_state_long = trial.suggest_categorical('d_state_long', [32, 64, 128])
    d_state_short = trial.suggest_categorical('d_state_short', [8, 16, 32])
    long_patch = trial.suggest_categorical('long_patch', [6, 12, 24])  # 30/60/120min
    short_patch = trial.suggest_categorical('short_patch', [1, 2, 4])  # 5/10/20min
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 5e-4, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-4, 1e-2, log=True)
    
    # ProfitWeightedCE params
    profit_scale = trial.suggest_float('profit_scale', 5.0, 100.0, log=True)
    direction_penalty = trial.suggest_float('direction_penalty', 1.0, 4.0)
    min_weight = trial.suggest_float('min_weight', 0.1, 0.5)
    max_weight = trial.suggest_float('max_weight', 5.0, 20.0)
    
    # Scoring weights
    acc_weight = trial.suggest_float('acc_weight', 0.5, 2.0)
    baseline_margin_weight = trial.suggest_float('baseline_margin_weight', 0.5, 2.0)
    
    # --- Setup DataLoaders ---
    # Use fixed LONG_LOOKBACK for data loading
    try:
        train_loader, val_loader = create_dataloaders(
            df_prepared,
            FEATURE_COLS,
            long_lookback=LONG_LOOKBACK,  # Fixed at 256
            batch_size=batch_size,
            val_split=0.2,
            enforce_contiguous=True,
        )
    except Exception as e:
        print(f"Dataloader failed: {e}")
        return -1e9
    
    # --- Model with Dual Lookbacks ---
    model = DualLookbackHierarchicalMamba(
        input_dim=len(FEATURE_COLS),
        d_model=d_model,
        long_lookback=LONG_LOOKBACK,   # Fixed at 256 bars
        long_patch=long_patch,
        short_lookback=short_lookback,  # Tunable: 32-128 bars
        short_patch=short_patch,
        n_long_layers=n_long_layers,
        n_short_layers=n_short_layers,
        d_state_long=d_state_long,
        d_state_short=d_state_short,
        num_classes=NUM_CLASSES,
        dropout=dropout,
    ).to(device)
    
    trial.set_user_attr("n_params", sum(p.numel() for p in model.parameters()))
    trial.set_user_attr("long_tokens", model.get_token_counts()['long_tokens'])
    trial.set_user_attr("short_tokens", model.get_token_counts()['short_tokens'])
    
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
    
    criterion = ProfitWeightedCE(
        profit_scale=profit_scale,
        min_weight=min_weight,
        max_weight=max_weight,
        direction_penalty=direction_penalty,
    )
    
    # --- Training ---
    best_score = float('-inf')
    best_metrics = {}
    patience = 10
    patience_counter = 0
    
    for epoch in range(20):
        train_loss, train_acc, train_pnl = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc, val_pnl, max_class_pct = evaluate(
            model, val_loader, criterion, device
        )
        
        scheduler.step()
        
        # Compute score
        score = compute_accuracy_weighted_pnl(
            val_acc, val_pnl, max_class_pct,
            acc_weight=acc_weight,
            baseline_margin_weight=baseline_margin_weight,
        )
        
        if score > best_score:
            best_score = score
            patience_counter = 0
            best_metrics = {
                "final_acc": val_acc,
                "final_loss": val_loss,
                "final_pnl": val_pnl,
                "max_class_pct": max_class_pct,
                "acc_margin": val_acc - max_class_pct,
                "acc_weighted_score": score,
            }
        else:
            patience_counter += 1
        
        print(f"  E{epoch+1:02} | Acc: {val_acc:.1f}% (base: {max_class_pct:.1f}%) | "
              f"PnL: {val_pnl:.6f} | Score: {score:.6f}")
        
        trial.report(score, epoch)
        
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        if patience_counter >= patience:
            break
    
    for key, value in best_metrics.items():
        trial.set_user_attr(key, value)
    
    return best_score

In [ ]:
# Optimization settings
N_TRIALS = 30
STUDY_NAME = f"dual_lookback_mamba_{TICKER}_clf"

# Create study
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)

print(f"Starting optimization: {N_TRIALS} trials")
print(f"Ticker: {TICKER}")
print(f"Features: {len(FEATURE_COLS)}")
print(f"Architecture: DualLookbackHierarchicalMamba")
print(f"  Long lookback: {LONG_LOOKBACK} bars (fixed)")
print(f"  Short lookback: {SHORT_LOOKBACK_MIN}-{SHORT_LOOKBACK_MAX} bars (tunable)")
print(f"Objective: Accuracy-Weighted PnL")
print("-" * 60)

In [ ]:
# Run optimization
study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
)

## 8. Analyze Results

In [ ]:
# Best trial summary
print("\n" + "=" * 60)
print("BEST TRIAL SUMMARY")
print("=" * 60)

best = study.best_trial
acc = best.user_attrs.get('final_acc', 0)
max_class = best.user_attrs.get('max_class_pct', 33.33)
acc_margin = best.user_attrs.get('acc_margin', acc - max_class)

print(f"\nAccuracy-Weighted PnL Score: {best.value:.6f}")
print(f"\n--- Accuracy Breakdown ---")
print(f"  Accuracy: {acc:.2f}%")
print(f"  Baseline (max class): {max_class:.2f}%")
print(f"  Margin over baseline: {acc_margin:+.2f}%")
print(f"\n--- Trading Metrics ---")
print(f"  Raw PnL: {best.user_attrs.get('final_pnl', 0):.6f}")
print(f"\n--- Model Info ---")
print(f"  Parameters: {best.user_attrs.get('n_params', 0):,}")
print(f"  Long tokens: {best.user_attrs.get('long_tokens', 0)}")
print(f"  Short tokens: {best.user_attrs.get('short_tokens', 0)}")
print(f"\n--- Key Hyperparameters ---")
for k in ['d_model', 'short_lookback', 'long_patch', 'short_patch', 'n_long_layers', 'n_short_layers']:
    if k in best.params:
        print(f"  {k}: {best.params[k]}")
print(f"  long_lookback: {LONG_LOOKBACK} (fixed)")

In [ ]:
# Visualization
trials_df = study.trials_dataframe()
completed = trials_df[trials_df['state'] == 'COMPLETE'].copy()
print(f"Completed trials: {len(completed)}/{N_TRIALS}")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Score Distribution
ax = axes[0, 0]
ax.hist(completed['value'], bins=20, alpha=0.7, color='blue', edgecolor='black')
ax.axvline(best.value, color='red', linestyle='--', linewidth=2, label=f'Best: {best.value:.4f}')
ax.set_xlabel('Accuracy-Weighted PnL')
ax.set_ylabel('Count')
ax.set_title('Score Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Optimization History
ax = axes[0, 1]
ax.plot(completed['number'], completed['value'], 'o-', alpha=0.7)
ax.set_xlabel('Trial')
ax.set_ylabel('Score')
ax.set_title('Optimization History')
ax.grid(True, alpha=0.3)

# 3. d_model vs Score
ax = axes[0, 2]
if 'params_d_model' in completed.columns:
    ax.scatter(completed['params_d_model'], completed['value'], alpha=0.6, s=80)
ax.set_xlabel('d_model')
ax.set_ylabel('Score')
ax.set_title('Model Size vs Performance')
ax.grid(True, alpha=0.3)

# 4. Short Lookback vs Score (KEY PARAMETER)
ax = axes[1, 0]
if 'params_short_lookback' in completed.columns:
    ax.scatter(completed['params_short_lookback'], completed['value'], alpha=0.6, s=80, c='green')
ax.set_xlabel('Short Lookback (bars)')
ax.set_ylabel('Score')
ax.set_title('Short Lookback vs Performance')
ax.grid(True, alpha=0.3)

# 5. Long Patch vs Score
ax = axes[1, 1]
if 'params_long_patch' in completed.columns:
    ax.scatter(completed['params_long_patch'], completed['value'], alpha=0.6, s=80)
ax.set_xlabel('Long Patch Size (bars)')
ax.set_ylabel('Score')
ax.set_title('Long Patch vs Performance')
ax.grid(True, alpha=0.3)

# 6. Learning Rate vs Score
ax = axes[1, 2]
if 'params_learning_rate' in completed.columns:
    ax.scatter(completed['params_learning_rate'], completed['value'], alpha=0.6, s=80)
    ax.set_xscale('log')
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Score')
ax.set_title('Learning Rate vs Performance')
ax.grid(True, alpha=0.3)

plt.suptitle(f"DualLookbackHierarchicalMamba Optimization ({TICKER})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{STUDY_NAME}_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Parameter importance
fig, ax = plt.subplots(figsize=(10, 6))

try:
    importances = optuna.importance.get_param_importances(study)
    params = list(importances.keys())[:12]
    values = [importances[p] for p in params]
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(params)))
    ax.barh(params, values, color=colors)
    ax.set_xlabel('Importance')
    ax.set_title('Hyperparameter Importance')
    ax.grid(True, alpha=0.3, axis='x')
except Exception as e:
    ax.text(0.5, 0.5, f'Not enough trials\n{e}',
            ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Hyperparameter Importance')

plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{STUDY_NAME}_importance.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Train Final Model

In [ ]:
# Train final model with best parameters
NUM_EPOCHS = 40

params = study.best_params
print(f"Training final model with best parameters:")
print(json.dumps(params, indent=2))

# Create dataloaders with fixed long lookback
train_loader, val_loader = create_dataloaders(
    df_prepared,
    FEATURE_COLS,
    long_lookback=LONG_LOOKBACK,  # Fixed at 256
    batch_size=params['batch_size'],
    val_split=0.2,
    enforce_contiguous=True,
)

# Create model with dual lookbacks
final_model = DualLookbackHierarchicalMamba(
    input_dim=len(FEATURE_COLS),
    d_model=params['d_model'],
    long_lookback=LONG_LOOKBACK,          # Fixed at 256 bars
    long_patch=params['long_patch'],
    short_lookback=params['short_lookback'],  # From optimization
    short_patch=params['short_patch'],
    n_long_layers=params['n_long_layers'],
    n_short_layers=params['n_short_layers'],
    d_state_long=params['d_state_long'],
    d_state_short=params['d_state_short'],
    num_classes=NUM_CLASSES,
    dropout=params['dropout'],
).to(device)

print(f"\nModel parameters: {sum(p.numel() for p in final_model.parameters()):,}")
print(f"Token counts: {final_model.get_token_counts()}")

optimizer = optim.AdamW(
    final_model.parameters(),
    lr=params['learning_rate'],
    weight_decay=params['weight_decay']
)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

criterion = ProfitWeightedCE(
    profit_scale=params['profit_scale'],
    min_weight=params['min_weight'],
    max_weight=params['max_weight'],
    direction_penalty=params['direction_penalty'],
)

In [ ]:
# Training loop
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_pnl': []}
best_score = float('-inf')
best_state = None

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc, train_pnl = train_epoch(
        final_model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc, val_pnl, max_class_pct = evaluate(
        final_model, val_loader, criterion, device
    )
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_pnl'].append(val_pnl)
    
    score = compute_accuracy_weighted_pnl(
        val_acc, val_pnl, max_class_pct,
        acc_weight=params.get('acc_weight', 1.0),
        baseline_margin_weight=params.get('baseline_margin_weight', 1.0),
    )
    
    if score > best_score:
        best_score = score
        best_state = final_model.state_dict().copy()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02}/{NUM_EPOCHS} | "
              f"Acc: {val_acc:.1f}% (base: {max_class_pct:.1f}%) | "
              f"PnL: {val_pnl:.6f} | Score: {score:.6f}")

# Load best state
if best_state:
    final_model.load_state_dict(best_state)
print(f"\nBest score: {best_score:.6f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(history['train_loss'], label='Train', alpha=0.7)
ax.plot(history['val_loss'], '--', label='Val', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(history['train_acc'], label='Train', alpha=0.7)
ax.plot(history['val_acc'], '--', label='Val', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(history['val_pnl'], color='green', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation PnL')
ax.set_title('Validation PnL')
ax.grid(True, alpha=0.3)

plt.suptitle(f"Final Model Training ({TICKER})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{STUDY_NAME}_training.png", dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Artifacts

In [ ]:
import joblib

# Save study
joblib.dump(study, RESULTS_PATH / f"{STUDY_NAME}_study.pkl")

# Save trial data
trials_df.to_csv(RESULTS_PATH / f"{STUDY_NAME}_trials.csv", index=False)

# Save best params with dual lookback config
best_params = {
    **study.best_params,
    'best_value': study.best_value,
    'ticker': TICKER,
    'task': TASK,
    'num_classes': NUM_CLASSES,
    'feature_cols': FEATURE_COLS,
    'target_col': TARGET_COL,
    'clf_percentiles': CLF_PERCENTILES,
    'q_low': float(q_low),
    'q_high': float(q_high),
    # Dual lookback architecture
    'long_lookback': LONG_LOOKBACK,
    'long_tokens': final_model.get_token_counts()['long_tokens'],
    'short_tokens': final_model.get_token_counts()['short_tokens'],
}
with open(RESULTS_PATH / f"{STUDY_NAME}_best_params.json", 'w') as f:
    json.dump(best_params, f, indent=2)

# Save final model
torch.save({
    'model_state_dict': final_model.state_dict(),
    'params': study.best_params,
    'history': history,
    'feature_cols': FEATURE_COLS,
    'clf_thresholds': {'q_low': q_low, 'q_high': q_high},
    'architecture': {
        'long_lookback': LONG_LOOKBACK,
        'short_lookback': params['short_lookback'],
        'long_patch': params['long_patch'],
        'short_patch': params['short_patch'],
    },
}, RESULTS_PATH / f"{STUDY_NAME}_final_model.pth")

print(f"\nArtifacts saved to: {RESULTS_PATH}")
for f in RESULTS_PATH.glob(f"{STUDY_NAME}*"):
    print(f"  - {f.name}")

In [ ]:
# Final summary
print("\n" + "=" * 60)
print("EXPERIMENT COMPLETE")
print("=" * 60)
print(f"\nTicker: {TICKER}")
print(f"Task: {NUM_CLASSES}-class classification (terciles)")
print(f"Features: {len(FEATURE_COLS)}")
print(f"Target: {TARGET_COL} ({TARGET_STEPS * BAR_MINUTES}min forward return)")
print(f"\nTrials: {len(completed)}/{N_TRIALS}")
print(f"\nBest Trial:")
print(f"  Score: {study.best_value:.6f}")
print(f"  Accuracy: {best.user_attrs.get('final_acc', 0):.1f}%")
print(f"  Baseline: {best.user_attrs.get('max_class_pct', 33.33):.1f}%")
print(f"  PnL: {best.user_attrs.get('final_pnl', 0):.6f}")
print(f"\nDual Lookback Architecture:")
print(f"  d_model: {params['d_model']}")
print(f"  Long branch: {params['n_long_layers']} layers, lookback={LONG_LOOKBACK}, patch={params['long_patch']}")
print(f"    -> {LONG_LOOKBACK // params['long_patch']} tokens")
print(f"  Short branch: {params['n_short_layers']} layers, lookback={params['short_lookback']}, patch={params['short_patch']}")
print(f"    -> {(params['short_lookback'] - params['short_patch']) // 1 + 1} tokens")